In [3]:
import pandas as pd
from sqlalchemy import create_engine
from config import DB_HOST, DB_PORT, DB_USER, DB_NAME, DB_PASSWORD


DATABASE_URL = (
    f"postgresql+psycopg://"
    f"{DB_USER}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/"
    f"{DB_NAME}"
)

engine = create_engine(DATABASE_URL)
df = pd.read_sql("SELECT * FROM food_prices ORDER BY date;", engine)

In [1]:
import random
import os
import numpy as np

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(SEED)

In [5]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 16372 entries, 0 to 16371
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   variant_id    16372 non-null  int64 
 1   variant_name  16372 non-null  str   
 2   unit          16372 non-null  str   
 3   date          16372 non-null  object
 4   region        16372 non-null  str   
 5   price         16372 non-null  int64 
dtypes: int64(2), object(1), str(3)
memory usage: 767.6+ KB
None


In [ ]:
print(df.isna().sum().sum(), "missing values")
print(df.duplicated().sum(), "duplicates")
print(df['variant_name'].nunique(), "unique items")
print(df['region'].nunique(), "unique region")
print(df['date'].min(), "-", df['date'].max(), "data")

date_range = pd.date_range(df['date'].min(), df['date'].max())
missing_dates = set(date_range.date) - set(pd.to_datetime(df['date']).dt.date)

print(len(missing_dates), "missing dates out of", len(date_range), "total days")

0 missing values
52 duplicates
17 unique items
4 unique region
2024-02-01 - 2026-09-08 data
306 missing dates out of 951 total days


In [13]:
df = df.drop_duplicates()
df['date'] = pd.to_datetime(df['date'])

print(df.isna().sum().sum(), "missing values")

0 missing values


In [14]:
df['year_month'] = df['date'].dt.to_period('M')

monthly = df.groupby(['variant_id', 'variant_name', 'region', 'year_month'])['price'].mean().reset_index()
monthly = monthly.rename(columns={'price' : 'avg_price'})
monthly = monthly.sort_values(['variant_id', 'region', 'year_month'])

In [16]:
print(monthly.shape)
print(monthly.head(20))

(824, 5)
    variant_id      variant_name    region year_month     avg_price
0            1  Beras SPHP Bulog  National    2024-02  11571.750000
1            1  Beras SPHP Bulog  National    2024-03  11720.809524
2            1  Beras SPHP Bulog  National    2024-04  11289.321429
3            1  Beras SPHP Bulog  National    2024-05  11822.304348
4            1  Beras SPHP Bulog  National    2024-06  12266.888889
5            1  Beras SPHP Bulog  National    2024-07  12315.409091
6            1  Beras SPHP Bulog  National    2024-08  12329.636364
7            1  Beras SPHP Bulog  National    2024-09  12347.800000
8            1  Beras SPHP Bulog  National    2024-10  12324.478261
9            1  Beras SPHP Bulog  National    2024-11  12295.900000
10           1  Beras SPHP Bulog  National    2024-12  12304.900000
11           1  Beras SPHP Bulog  National    2025-01  12288.157895
12           1  Beras SPHP Bulog  National    2025-02  12272.500000
13           1  Beras SPHP Bulog  Natio

In [43]:
import numpy as np

group = monthly.groupby(['variant_id', 'region'])['avg_price']

monthly['price_log'] = np.log1p(monthly['avg_price'])
group_log = monthly.groupby(['variant_id', 'region'])['price_log']

monthly['price_lag1'] = group_log.shift(1)
monthly['price_lag2'] = group_log.shift(2)
monthly['price_lag12'] = group_log.shift(12)
monthly['price_roll3'] = group_log.transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
monthly['price_roll12'] = group_log.transform(lambda x: x.shift(1).rolling(12, min_periods=2).mean())
monthly['momentum'] = monthly['price_lag1'] - monthly['price_roll3']

monthly['target_next_month'] = group_log.shift(-1)

In [44]:
monthly.isna().sum()

variant_id             0
variant_name           0
region                 0
year_month             0
avg_price              0
price_log              0
price_lag1            26
price_lag12          312
price_roll3           52
momentum              52
target_next_month     26
price_lag2            52
price_roll12          52
dtype: int64

In [61]:
model_data = monthly.dropna(subset=['price_lag1', 'target_next_month']).copy()
model_data = pd.get_dummies(model_data, columns=['variant_name', 'region'], prefix=['item', 'region'])

features = ['price_lag1', 'price_lag2', 'price_lag12', 'price_roll3', 'price_roll12', 'momentum'] + \
                        [c for c in model_data.columns if c.startswith('item_') or c.startswith('region_')]

print(model_data[['target_next_month', 'price_lag1', 'price_lag2', 'price_lag12', 'price_roll12', 'price_roll3', 'momentum']].corr()['target_next_month'])

target_next_month    1.000000
price_lag1           0.984551
price_lag2           0.978940
price_lag12          0.977459
price_roll12         0.985633
price_roll3          0.982399
momentum             0.034688
Name: target_next_month, dtype: float64


In [100]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_data = model_data.sort_values('year_month')
model_data['change_next_month'] = model_data['target_next_month'] - model_data['price_lag1']

#time based split to avoid leaking
split_point = model_data['year_month'].quantile(0.85)
train = model_data[model_data['year_month'] < split_point]
val = model_data[model_data['year_month'] >= split_point]

X_train = train[features]
y_train = train['change_next_month']
X_val = val[features]

model = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

predicted_change = model.predict(X_val)
final_pred = val['price_lag1'].values + predicted_change
final_pred_real = np.expm1(final_pred)
actual_real = np.expm1(val['target_next_month'])

mae = mean_absolute_error(actual_real, final_pred_real)
r2 = r2_score(actual_real, final_pred_real)

naive_pred = np.expm1(val['price_lag1'])
naive_mae = mean_absolute_error(actual_real, naive_pred)

print("Monthly model MAE:", mae, "R²:", r2)
print("Naive baseline MAE:", naive_mae)

Monthly model MAE: 1221.2116102730954 R²: 0.9864333660538375
Naive baseline MAE: 1597.9194550129473


In [94]:
importances = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(importances.head(8))

                   feature  importance
0               price_lag1    0.233396
4             price_roll12    0.224199
1               price_lag2    0.152801
5                 momentum    0.114806
3              price_roll3    0.072488
2              price_lag12    0.066850
13  item_Cabai Rawit Merah    0.049032
6        item_Bawang Merah    0.044649


In [101]:
import joblib


X_full = model_data[features]
y_full= model_data['change_next_month']

final_model = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42, n_jobs=-1)
final_model.fit(X_full, y_full)

joblib.dump(final_model, 'price_forecast_model_monthly_2.pkl')
joblib.dump(features, 'feature_columns.pkl')

print("Saved monthly model and features.")

Saved monthly model and features.
